# Kyiv Telegram Weapons Scraper → piterfm-format CSV (+ exact timestamps)

Collects weapon mentions for **Kyiv** from Telegram monitoring channels and writes a CSV in
the **same schema as the piterfm "Massive Missile Attacks" dataset** (`missile_attacks_daily.csv`),
**plus two extra columns**: `message_posted_at` (the exact time the post appeared, UTC) and
`num_targets` (count parsed from the message). This gives the per-event time resolution the
night-level Kaggle data lacks, so weapons can be matched to individual Kyiv alerts.

### Channels (configurable — swap in whatever you trust)
1. **`@kpszsu`** — *official* Air Force of Ukraine. Authoritative weapon **types, counts and
   launch directions**, plus per-group "course on Kyiv oblast / district" updates and morning
   summaries. This is the backbone.
2. **`@air_alert_ua`** — *official* "Повітряна тривога" alert channel. Precise **alert start/stop**
   per region (useful to bound weapon events to Kyiv alert windows).
3. **`@napramok`** — a high-volume *unofficial* monitoring channel posting per-group threat type
   and direction. **Verify the exact handle yourself, treat it as unofficial, and cross-check
   against `@kpszsu`.**

> ⚠️ **Use responsibly.** Ukrainian authorities warn against anonymous channels that publish
> *precise real-time trajectories* (they can help the enemy correct aim), and **fake Air-Force
> impersonator channels exist** — confirm `@kpszsu` is the real one. This notebook is for
> **retrospective** analysis of historical posts, not building a live tracking feed. Prefer the
> official channels; the unofficial one is a cross-check only.

## Setup

In [8]:
# Telethon talks to Telegram's API using YOUR account. Get api_id/api_hash at https://my.telegram.org
# (Settings → API development tools). Never commit these.
# !pip install telethon --quiet
import os, re, csv, asyncio, datetime as dt
from pathlib import Path
import pandas as pd

from telethon import TelegramClient

API_ID   = int(os.environ.get("TG_API_ID", "34081429"))      # set TG_API_ID / TG_API_HASH in your env
API_HASH = os.environ.get("TG_API_HASH", "---") # I don't add hash
SESSION  = "kyiv_scraper"                               # local session file (also keep out of git)

CHANNELS  = ["kpszsu", "napramok"]      # edit freely
DATE_FROM = dt.datetime(2025, 6, 24, tzinfo=dt.timezone.utc)
DATE_TO   = dt.datetime(2026, 6, 24, tzinfo=dt.timezone.utc)
OUT_CSV   = Path("kyiv_telegram_weapons.csv")

# Kyiv relevance: city, oblast, and the city's districts (raions).
KYIV_RE = re.compile(
    r"ки[їє]в|kyiv|kiev|столиц|оболон|печерськ|подільськ|святошин|дарниц|"
    r"деснянськ|дніпровськ|солом'?янськ|шевченківськ|голосіївськ|вишгород|бровар", re.I)
print("Config ready. Channels:", CHANNELS)

Config ready. Channels: ['kpszsu', 'napramok']


## Weapon parser

Pure-text extraction tuned to how these channels write (informal Ukrainian/Russian + slang).
It returns, per message: the weapon **classes** present, a representative **model** string, an
**`optical_cruise`** flag (Kh-101 / Kh-555 family — the missiles whose optical scene-matching
corrector is the subject of the clear-sky hypothesis), and a **target count**.

In [2]:
# Term dictionaries (lowercase; matched case-insensitively). Cover en / uk / ru / slang.
TERMS = {
 "shahed":   ["шахед","shahed","геран","герань","geran","мопед","бпла","дрон","uav",
              "ударних бпла","безпілотник"],
 "decoy":    ["гербера","gerbera","пародія","parody","імітатор","decoy","італмас","пародии"],
 "cruise":   ["крилат","крылат","калібр","калибр","х-101","kh-101","х-555","kh-555",
              "х-59","kh-59","х-22","kh-22"],
 "ballistic":["баліст","баллист","іскандер","искандер","кинжал","кинджал","кинзал",
              "kh-47","х-47","кн-23","kn-23","міг-31","mig-31","міг31","зліт міг"],
 "kab":      ["каб","kab","авіабомб","керована авіабомба","фаб"],
}
OPTICAL_CRUISE = ["х-101","kh-101","х-555","kh-555"]    # optical scene-matching family

MODEL_PATTERNS = [   # representative model string when explicitly named
 (r"х-?101|kh-?101", "Kh-101"), (r"х-?555|kh-?555", "Kh-555"),
 (r"калібр|калибр|kalibr", "Kalibr"), (r"іскандер-?к|iskander-?k", "Iskander-K"),
 (r"іскандер-?м|искандер|iskander-?m", "Iskander-M"),
 (r"кинж|кинд|кинзал|kh-?47|х-?47", "Kinzhal"), (r"кн-?23|kn-?23", "KN-23"),
 (r"гербера|gerbera", "Gerbera (decoy)"), (r"шахед|shahed|геран|geran", "Shahed-136"),
]

NUM_WORDS = {"один":1,"одна":1,"одну":1,"два":2,"дві":2,"три":3,"чотири":4,"п'ять":5,
             "пять":5,"шість":6,"сім":7,"вісім":8,"дев'ять":9,"десять":10}

def classify_text(text):
    t = (text or "").lower()
    classes = {c for c, kws in TERMS.items() if any(k in t for k in kws)}
    # MiG-31 takeoff is a ballistic (Kinzhal) precursor signal even without the word 'ballistic'
    if "міг-31" in t or "mig-31" in t or "зліт міг" in t:
        classes.add("ballistic")
    optical = any(k in t for k in OPTICAL_CRUISE)
    model = next((label for pat, label in MODEL_PATTERNS if re.search(pat, t)), None)
    return classes, model, optical

def count_targets(text):
    t = (text or "").lower()
    # Strip weapon model designations (Kh-101, KN-23, MiG-31, S-300, X-59 …) and clock times,
    # so their digits aren't mistaken for a target count.
    t = re.sub(r"(?:kh|х|x|кн|kn|міг|mig|с-|s-)\s*-?\s*\d{1,3}", " ", t)
    t = re.sub(r"\b\d{1,2}:\d{2}\b", " ", t)
    nums = [int(n) for n in re.findall(r"\b(\d{1,3})\b", t)]
    nums = [n for n in nums if n <= 500]                 # drop years / huge irrelevancies
    if nums:
        return max(nums)                                 # e.g. "5 цілей", "135 ударними бпла"
    for w, v in NUM_WORDS.items():                       # spelled-out small numbers
        if re.search(rf"\b{w}\b", t):
            return v
    return None                                          # 'група' with no number → unknown

## Scraper (Telethon)

In [3]:
async def scrape_channel(client, username, date_from, date_to):
    # Pull message history in [date_from, date_to] and parse weapon mentions for Kyiv.
    rows = []
    async for m in client.iter_messages(username, offset_date=date_to, reverse=False):
        if m.date < date_from:
            break                                        # iter_messages goes newest→oldest
        text = m.message or ""
        if not KYIV_RE.search(text):
            continue                                     # keep only Kyiv-relevant posts
        classes, model, optical = classify_text(text)
        if not classes:
            continue                                     # no weapon mention → skip
        rows.append({
            # ---- piterfm-compatible columns ----
            "time_start":  m.date.replace(tzinfo=None),  # event time proxy (post time, UTC naive)
            "time_end":    None,
            "model":       model or ",".join(sorted(classes)),
            "launched":    count_targets(text),          # ≈ # targets reported in the post
            "destroyed":   None,                         # monitoring posts rarely confirm kills
            "target":      "Kyiv",
            "launch_place": None,
            # ---- extra columns requested ----
            "message_posted_at": m.date.isoformat(),     # EXACT post timestamp (UTC)
            "num_targets":  count_targets(text),
            "has_shahed":   "shahed" in classes,
            "has_cruise":   "cruise" in classes,
            "has_ballistic":"ballistic" in classes,
            "has_decoy":    "decoy" in classes,
            "optical_cruise": optical,
            "channel":      username,
            "message_id":   m.id,
            "edit_date":    m.edit_date.isoformat() if m.edit_date else None,
            "raw_text":     text.replace("\n", " ").strip()[:500],
        })
    print(f"   {username}: {len(rows)} Kyiv weapon posts")
    return rows

In [4]:
from telethon.errors import (FloodWaitError, ApiIdInvalidError,
    PhoneNumberInvalidError, PhoneNumberBannedError, PhoneNumberUnoccupiedError)

PHONE = input("Phone, full international (+countrycode, no spaces): ").strip()

client = TelegramClient(SESSION, API_ID, API_HASH)
await client.connect()
print("connected:", client.is_connected(),
      "| already authorized:", await client.is_user_authorized())

try:
    sent = await client.send_code_request(PHONE)
    print("✅ code request accepted")
    print("   delivered via :", type(sent.type).__name__)   # App / Sms / Call / FlashCall
    print("   next resend   :", type(sent.next_type).__name__ if sent.next_type else None)
    print("   timeout (s)   :", sent.timeout)
except FloodWaitError as e:
    print(f"⛔ FLOOD WAIT — Telegram is throttling you. Wait {e.seconds}s "
          f"(~{e.seconds//60} min) and do NOT retry before then.")
except ApiIdInvalidError:
    print("⛔ api_id / api_hash invalid or mismatched — re-copy BOTH from my.telegram.org.")
except PhoneNumberInvalidError:
    print("⛔ phone format rejected — must be +<country code><number>, e.g. +380...")
except PhoneNumberBannedError:
    print("⛔ this number is banned on Telegram.")
except PhoneNumberUnoccupiedError:
    print("⛔ no Telegram account exists on this number — register it in the app first.")

Phone, full international (+countrycode, no spaces):  +380962878411


connected: True | already authorized: False
✅ code request accepted
   delivered via : SentCodeTypeApp
   next resend   : None
   timeout (s)   : None


Server closed the connection: 0 bytes read on a total of 8 expected bytes
Error executing high-level request after reconnect: <class 'sqlite3.OperationalError'>: database is locked
Server closed the connection: 0 bytes read on a total of 8 expected bytes
Error executing high-level request after reconnect: <class 'sqlite3.OperationalError'>: database is locked
Server closed the connection: 0 bytes read on a total of 8 expected bytes
Task exception was never retrieved
future: <Task finished name='Task-286' coro=<UpdateMethods._dispatch_update() done, defined at /home/homo-novus/miniconda3/envs/my_env/lib/python3.10/site-packages/telethon/client/updates.py:556> exception=OperationalError('database is locked')>
Traceback (most recent call last):
  File "/home/homo-novus/miniconda3/envs/my_env/lib/python3.10/site-packages/telethon/client/updates.py", line 568, in _dispatch_update
    await self.get_me(input_peer=True)
  File "/home/homo-novus/miniconda3/envs/my_env/lib/python3.10/site-packa

In [5]:
# pip install "qrcode[pil]"
import qrcode, asyncio, getpass
from telethon.errors import SessionPasswordNeededError

async def qr_login():
    client = TelegramClient(SESSION, API_ID, API_HASH)
    await client.connect()
    if await client.is_user_authorized():
        print("Already authorized — no QR needed."); await client.disconnect(); return
    qr = await client.qr_login()
    while True:
        qrcode.make(qr.url).save("login_qr.png")
        print("Scan login_qr.png →  phone Telegram → Settings → Devices → Link Desktop Device")
        try:
            await qr.wait(timeout=30); break          # waits for the scan
        except asyncio.TimeoutError:
            await qr.recreate()                        # QR expired → refresh it
        except SessionPasswordNeededError:
            await client.sign_in(password=getpass.getpass("2FA password: ")); break
    print("Authorized:", await client.is_user_authorized())
    await client.disconnect()

await qr_login()



Scan login_qr.png →  phone Telegram → Settings → Devices → Link Desktop Device
Authorized: True


In [9]:
async def main():
    if not API_ID or not API_HASH:
        raise RuntimeError("Set TG_API_ID and TG_API_HASH (from https://my.telegram.org) first.")
    client = TelegramClient(SESSION, API_ID, API_HASH)
    await client.connect()                       # connect only — does NOT trigger login
    if not await client.is_user_authorized():
        await client.disconnect()
        raise RuntimeError("Session not authorized yet — run the QR-login cell (qr_login) first.")
    try:
        all_rows = []
        for ch in CHANNELS:
            try:
                all_rows += await scrape_channel(client, ch, DATE_FROM, DATE_TO)
            except Exception as e:
                print(f"   ⚠️  {ch} failed: {e}")
        return all_rows
    finally:
        await client.disconnect()

## Run & save

In [10]:
# ── Run the scraper ──────────────────────────────────────────────────────────
# Jupyter already runs an asyncio loop, so AWAIT main() directly (no run_until_complete).
# FIRST RUN is interactive: Telethon will prompt for your phone number, then the login
# code Telegram sends you (and your 2FA password if you have one). After that the local
# `kyiv_scraper.session` file remembers you and it won't ask again.
import csv

rows = await main()            # ← the key change.  (.py script alternative: asyncio.run(main()))

df = pd.DataFrame(rows).sort_values("message_posted_at")
df.to_csv(OUT_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"\n✅ Saved {len(df):,} rows → {OUT_CSV}")
df.head()

   kpszsu: 1959 Kyiv weapon posts
   napramok: 0 Kyiv weapon posts

✅ Saved 1,959 rows → kyiv_telegram_weapons.csv


,time_start,time_end,model,launched,destroyed,target,launch_place,message_posted_at,num_targets,has_shahed,has_cruise,has_ballistic,has_decoy,optical_cruise,channel,message_id,edit_date,raw_text
1958,2025-06-26 19:03:43,None,shahed,NaN,None,Kyiv,None,2025-06-26T19:03:43+00:00,NaN,True,False,False,False,False,kpszsu,37086,2025-06-26T19:03:50+00:00,🛵 Нова група ворожих ударних БпЛА на півдні Су...
1957,2025-06-26 19:59:26,None,shahed,NaN,None,Kyiv,None,2025-06-26T19:59:26+00:00,NaN,True,False,False,False,False,kpszsu,37090,2025-06-26T19:59:35+00:00,🛵 Ворожі ударні БпЛА на Чернігівщині ➡️ курсом...
1956,2025-06-26 20:13:58,None,shahed,NaN,None,Kyiv,None,2025-06-26T20:13:58+00:00,NaN,True,False,False,False,False,kpszsu,37091,2025-06-26T20:14:07+00:00,🛵 Київ - ворожий БпЛА на околицях міста
1955,2025-06-26 20:44:17,None,shahed,NaN,None,Kyiv,None,2025-06-26T20:44:17+00:00,NaN,True,False,False,False,False,kpszsu,37092,2025-06-26T20:44:25+00:00,🛵 Нова група ударних БпЛА на сході Сумщини ➡️ ...
1954,2025-06-26 21:17:47,None,shahed,NaN,None,Kyiv,None,2025-06-26T21:17:47+00:00,NaN,True,False,False,False,False,kpszsu,37093,2025-06-26T21:17:59+00:00,🛵 Київ - ударні БпЛА в напрямку міста зі сход...


## Output schema

`kyiv_telegram_weapons.csv` columns:

* **piterfm-compatible** — `time_start` (post time used as the event-time proxy), `time_end`,
  `model`, `launched` (≈ targets reported), `destroyed`, `target`, `launch_place`.
* **added** — `message_posted_at` (exact UTC post time), `num_targets`, the class flags
  `has_shahed / has_cruise / has_ballistic / has_decoy`, `optical_cruise` (Kh-101/Kh-555 family),
  plus `channel`, `message_id`, `edit_date`, `raw_text`.

**Caveats baked in.** Monitoring posts report *detected / incoming* threats, so `launched` is an
observed count, not a confirmed launch tally, and `destroyed` is usually unknown. Post time leads
or lags the real event by minutes. `@kpszsu` morning summaries aggregate a whole night — those
rows carry the summary's timestamp, not per-group times; filter on the per-group posts when you
need exact timing. De-duplicate across channels (same event reported by several) before analysis.